# Round38 Full Workflow Notebook: `v38_5_chirps_delta_safe.csv`

这个 notebook 展示了 tied-best 阶段最重要的一个外部信号试探：在 `v37_5` 的 40/40 winner 上，使用 CHIRPS 月尺度降水异常只对湿月样本做 `EC` 轻量回退。

**为什么这本 notebook 重要**
- 它代表项目第一次把外部气候信号稳妥地嵌入后期 frontier。
- 变化范围很小，但验证了 `wet-month guard` 这条线在线上不会伤分。
- 这类 notebook 对公开读者最有价值，因为它清楚展示了 external data 怎么进入后处理。


## Step 1. Experiment Objective

**这个步骤做什么**
说明 round38 CHIRPS safe probe 的目标。

**为什么要做这个步骤**
外部数据试探的关键不是幅度大，而是让读者看清楚门控逻辑。

**预期产出**
得到该分支的策略角色说明。


**本分支摘要**
- Output file: `v38_5_chirps_delta_safe.csv`
- Base winner: `v37_5_ta40_ec40_push.csv`
- Relax source: `v37_4_ta40_ec35_push.csv`
- Gate: `if chirps_z >= wet_threshold then EC uses relaxed branch`
- TA / DRP: keep winner anchor


## Step 2. Environment Setup

**这个步骤做什么**
导入依赖并定义官方数据和公开 repo 资产地址。

**为什么要做这个步骤**
外部读者需要清楚看到 CHIRPS lookup 和锚点文件从哪里来。

**预期产出**
得到统一环境和远程数据读取函数。


In [ ]:
# =========================
# Step: Environment Setup
# 作用：导入依赖，定义官方数据源与公开 repo 资产地址
# =========================

from pathlib import Path
from io import StringIO
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 220)
pd.set_option('display.width', 220)

OFFICIAL_RAW = 'https://raw.githubusercontent.com/Snowflake-Labs/EY-AI-and-Data-Challenge/refs/heads/main'
SHOWCASE_RAW = 'https://raw.githubusercontent.com/FlalaGoGoGo/ey-water-2026-showcase/main'
OUTPUT_DIR = Path('generated_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)


def read_csv_remote(url: str) -> pd.DataFrame:
    """优先 pandas 直读，失败时回退到 curl，保证 notebook 更稳。"""
    try:
        return pd.read_csv(url)
    except Exception as exc:
        print(f'[warn] pandas direct read failed: {url}')
        print(f'       reason: {exc}')
        res = subprocess.run(['curl', '-L', url], check=True, capture_output=True, text=True)
        return pd.read_csv(StringIO(res.stdout))


## Step 3. Load Official Data, CHIRPS Lookup, And Anchors

**这个步骤做什么**
读取官方训练/验证数据、CHIRPS 月尺度缓存，以及两个 round37 提交锚点。

**为什么要做这个步骤**
这是把外部信号带入 notebook 的关键准备步骤。

**预期产出**
得到 round38 safe probe 所需的全部输入。


In [ ]:
train_wq = read_csv_remote(f'{OFFICIAL_RAW}/water_quality_training_dataset.csv')
val_tpl = read_csv_remote(f'{OFFICIAL_RAW}/submission_template.csv')
chirps = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/cache/chirps_monthly_lookup.csv')
winner_40_40 = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/targets/v37_5_ta40_ec40_push.csv')
relax_40_35 = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/anchors/round37/v37_4_ta40_ec35_push.csv')
reference_target = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/targets/v38_5_chirps_delta_safe.csv')

print('train_wq:', train_wq.shape)
print('val_tpl:', val_tpl.shape)
print('chirps:', chirps.shape)
print('winner_40_40:', winner_40_40.shape)
print('relax_40_35:', relax_40_35.shape)


## Step 4. Build CHIRPS Wet-Month Gate

**这个步骤做什么**
基于训练集与验证模板共同构造 `chirps_z`，再得到湿月 mask。

**为什么要做这个步骤**
round38 的关键不在模型，而在门控规则是否足够稳定。

**预期产出**
得到可以直接控制 EC 回退位置的 wet mask。


In [ ]:
def add_year_month(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['Sample Date'] = pd.to_datetime(out['Sample Date'], dayfirst=True, errors='coerce')
    out['year'] = out['Sample Date'].dt.year.astype(int)
    out['month'] = out['Sample Date'].dt.month.astype(int)
    return out

site_month_stats = (
    chirps.groupby(['Latitude', 'Longitude', 'month'])['chirps_monthly_precip']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'chirps_site_month_mean', 'std': 'chirps_site_month_std'})
)

train = add_year_month(train_wq).merge(chirps, on=['Latitude', 'Longitude', 'year', 'month'], how='left')
train = train.merge(site_month_stats, on=['Latitude', 'Longitude', 'month'], how='left')
train['chirps_site_month_std'] = train['chirps_site_month_std'].replace(0.0, np.nan)
train['chirps_anom'] = train['chirps_monthly_precip'] - train['chirps_site_month_mean']
train['chirps_z'] = train['chirps_anom'] / train['chirps_site_month_std']
wet_threshold = float(train['chirps_z'].dropna().quantile(0.80))

sub = add_year_month(val_tpl).merge(chirps, on=['Latitude', 'Longitude', 'year', 'month'], how='left')
sub = sub.merge(site_month_stats, on=['Latitude', 'Longitude', 'month'], how='left')
sub['chirps_site_month_std'] = sub['chirps_site_month_std'].replace(0.0, np.nan)
sub['chirps_anom'] = sub['chirps_monthly_precip'] - sub['chirps_site_month_mean']
sub['chirps_z'] = sub['chirps_anom'] / sub['chirps_site_month_std']
wet_mask = (sub['chirps_z'] >= wet_threshold).fillna(False).to_numpy()

print('wet_threshold_train_q80:', round(wet_threshold, 4))
print('wet_row_count:', int(wet_mask.sum()))
display(sub[['Latitude', 'Longitude', 'Sample Date', 'chirps_z']].head(5))


## Step 5. Render The CHIRPS Safe Branch

**这个步骤做什么**
在 40/40 winner 的基础上，仅对湿月行把 EC 回退到 40/35 版本。

**为什么要做这个步骤**
这样既保留了主 winner 的结构，又让外部降水信号只在小范围起作用。

**预期产出**
得到 round38 safe probe 的最终预测。


In [ ]:
ta = winner_40_40['Total Alkalinity'].to_numpy(dtype=float).copy()
ec = winner_40_40['Electrical Conductance'].to_numpy(dtype=float).copy()
drp = winner_40_40['Dissolved Reactive Phosphorus'].to_numpy(dtype=float).copy()

relax_ec = relax_40_35['Electrical Conductance'].to_numpy(dtype=float)
ec[wet_mask] = relax_ec[wet_mask]

submission = val_tpl[['Latitude', 'Longitude', 'Sample Date']].copy()
submission['Sample Date'] = pd.to_datetime(submission['Sample Date'], dayfirst=True, errors='coerce').dt.strftime('%d/%m/%Y')
submission['Total Alkalinity'] = np.clip(ta, 0, None)
submission['Electrical Conductance'] = np.clip(ec, 0, None)
submission['Dissolved Reactive Phosphorus'] = np.clip(drp, 0, None)
submission = submission[['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]
display(submission.head(3))


## Step 6. Export And Verify

**这个步骤做什么**
导出文件并和公开参考文件做 exact-match 检查。

**为什么要做这个步骤**
这样外部读者可以确认 notebook 不是“近似思路”，而是精确复现。

**预期产出**
得到 round38 safe probe 的复现校验结果。


In [ ]:
out_path = OUTPUT_DIR / 'v38_5_chirps_delta_safe.csv'
submission.to_csv(out_path, index=False)
print('saved to:', out_path.resolve())

metric_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
diff = (submission[metric_cols] - reference_target[metric_cols]).abs()
print('max abs diff by target:')
print(diff.max())
print('exact match:', bool((diff.max() < 1e-12).all()))


## Step 7. Diagnostics And Interpretation

**这个步骤做什么**
量化这次改动到底改了多少行，以及每行的平均 EC 变化。

**为什么要做这个步骤**
后期 tied-best 版本的特点就是“改动很少，但每个改动都有假设”。

**预期产出**
得到 round38 的可解释门控摘要。


In [ ]:
changed_mask = np.abs(winner_40_40['Electrical Conductance'] - submission['Electrical Conductance']) > 1e-12
summary = {
    'wet_row_count': int(wet_mask.sum()),
    'changed_row_count': int(changed_mask.sum()),
    'mean_ec_shift_on_changed_rows': float(np.mean(np.abs(winner_40_40.loc[changed_mask, 'Electrical Conductance'] - submission.loc[changed_mask, 'Electrical Conductance']))) if changed_mask.any() else 0.0,
}
print(summary)

plt.figure(figsize=(7, 4))
plt.hist(sub['chirps_z'].dropna(), bins=28, color='#ffd166', edgecolor='black', alpha=0.85)
plt.axvline(wet_threshold, color='#ffe600', linestyle='--', label='wet threshold')
plt.title('Validation CHIRPS z-score distribution')
plt.legend()
plt.show()


## Step 8. Interpretation

**这个步骤做什么**
把 CHIRPS safe probe 放回到后期 tied-best 平台里解释。

**为什么要做这个步骤**
让外部读者理解：外部数据不是一定要大幅度改预测，更多时候是作为风险门控。

**本分支结论**
- round38 的 CHIRPS probe 只改变了湿月样本的 EC。
- TA 与 DRP 保持 winner 锚点，这让变化幅度保持在安全区间。
- 该策略最终与 frontier 并列，证明 external signals 可以以 guard 的方式进入后期提交流程。
